# News Text Classification Task Using Natural Language Processing

**APOAI2025 Mock Competition Question 3**

The goal of the problem is to read a news article and predict its topic using the given training CSV dataset wherein unlike the basketball problem in question 1, the inputs here are words rather than coordinates. Because of this, the words must be converted into numbers.

# 1. The plan

I will be making the model through word embedding to represent words and a Long Short-Term Memory to process all of them in order. The general flow would be `reading the data, splitting it, creating a vocabulary, train and evaluate the model, then train a final model`.

# 2. Setup


The settings are grouped together so they are easier to inspect:

| Setting | Value | What it controls |
| --- | ---: | --- |
| Development epochs | 8 | Number of passes used to compare stopping points on validation data |
| Batch size | 64 | Maximum number of articles in one weight update |
| Learning rate | 0.001 | Step size used by Adam when adjusting parameters |
| Seed | 42 | Repeats the random initialization and data split |
| CPU threads | 4 | Number of CPU threads used by PyTorch |
| Maximum length | 128 | Maximum number of words kept per article |
| Vocabulary limit | 12,000 | Maximum number of entries, including padding and unknown IDs |

The following values are provided by an LLM's suggestion: https://chatgpt.com/share/6ad1a34b-5438-83ec-96a0-d63bff126d1c since I'm honestly not that great or familiar with picking the corect values for teh model's settings. These suggestions, however, does not guarantee it's correctness.

In [25]:
import csv
import json
import random
import re
from collections import Counter
from pathlib import Path
from types import SimpleNamespace

import torch
from torch import nn

roots = [Path.cwd(), *Path.cwd().parents]
candidates = [root / 'part3' for root in roots] + roots
PART_FOLDER = Path("/Users/davidfrancisco/Documents/ICAIC_Outputs/Question3")
TRAIN_PATH = PART_FOLDER / "train_v2" / "train_news.csv"
TEST_PATH = PART_FOLDER / "train_v2" / "test_news_nolabel.csv"
OUTPUT_DIR = PART_FOLDER

settings = SimpleNamespace(epochs=8, batch_size=64, learning_rate=0.001, seed=42,
                           threads=4, max_length=128, max_vocabulary=12000)
torch.set_num_threads(settings.threads)
torch.use_deterministic_algorithms(True)
print('Training CSV:', TRAIN_PATH)
print('Test CSV:', TEST_PATH)
print('Output folder:', OUTPUT_DIR)


Training CSV: /Users/davidfrancisco/Documents/ICAIC_Outputs/Question3/train_v2/train_news.csv
Test CSV: /Users/davidfrancisco/Documents/ICAIC_Outputs/Question3/train_v2/test_news_nolabel.csv
Output folder: /Users/davidfrancisco/Documents/ICAIC_Outputs/Question3


In [26]:
groups = {}

with TRAIN_PATH.open(newline='', encoding='utf-8-sig') as file:
    for row_number, row in enumerate(csv.DictReader(file), start=1):
        words = tuple(tokenize(row['text'], settings.max_length))
        groups.setdefault(words, []).append(
            (row_number, row['category'], row['text'])
        )
    

duplicates = [group for group in groups.values() if len(group) > 1]
conflicts = [
    group for group in duplicates
    if len({category for _, category, _ in group}) > 1
]

print("Repeated input groups:", len(duplicates))
print("Extra repeated rows:", sum(len(group) - 1 for group in duplicates))
print("Groups with conflicting labels:", len(conflicts))

print("\nFirst three repeated input groups:")
for group in duplicates[:3]:
    print([(row_number, category) for row_number, category, _ in group])

if conflicts:
    print("\nConflicting rows:")
    for group in conflicts:
        print("\nSame model input, different labels:")
        for row_number, category, text in group:
            print(f"Row {row_number} | Category: {category}")
            print(text[:200])
else:
    print("\nNo conflicting labels found.\n")

with TRAIN_PATH.open(newline='', encoding='utf-8-sig') as file:
    counts = Counter(row['category'] for row in csv.DictReader(file))

for category, count in sorted(counts.items()):
    print(f"{category}: {count} articles")

print(f"Total: {sum(counts.values())} articles")

Repeated input groups: 30
Extra repeated rows: 30
Groups with conflicting labels: 0

First three repeated input groups:
[(45, 'tech'), (570, 'tech')]
[(57, 'tech'), (818, 'tech')]
[(59, 'business'), (173, 'business')]

No conflicting labels found.

business: 290 articles
entertainment: 205 articles
sport: 284 articles
tech: 221 articles
Total: 1000 articles


# 2. Read the articles

The CSV has a text column containing the article and a category column containing its correct answer. We tokenize the article into smaller pieces so for instance we encounter "The Team Won!", then it gets split down to ['the', 'team', 'won']. I the sorted the categories' names and then assigned them specific IDs:

```text
business → 0, entertainment → 1, sport → 2, tech → 3
```

I also used a `seen` dictionary which makes sure that it keeps one copy of each distinct token sequence. I also already checked via the code cell above whether there are conflicting labels (same articles different categories). Through thorough checking of the repeated input groups, I was able to reduce the dataset to **970 distinct inputs**


In [27]:
def tokenize(text, max_length):
    return re.findall("[a-z]+(?:'[a-z]+)?", text.lower())[:max_length]

def read_training_data(path, max_length):
    with path.open(newline='', encoding='utf-8-sig') as handle:
        reader = csv.DictReader(handle)
        if not {'text', 'category'}.issubset(reader.fieldnames or []):
            raise ValueError('Training CSV must contain text and category columns')
        rows = list(reader)
    if not rows or any((not row['text'] or not row['category'] for row in rows)):
        raise ValueError('Training rows need nonempty text and category values')
    # Sort once so each category always has the same output position.
    categories = sorted({row['category'] for row in rows})
    category_ids = {name: index for index, name in enumerate(categories)}
    texts = []
    tokens = []
    labels = []
    seen = {}
    for row in rows:
        words = tokenize(row['text'], max_length)
        if not words:
            raise ValueError('A training article has no usable English words')
        # Compare the exact token sequences that the model will receive.
        key = tuple(words)
        if key in seen:
            if seen[key] != row['category']:
                raise ValueError('Identical model inputs have conflicting categories')
            continue
        seen[key] = row['category']
        texts.append(row['text'])
        tokens.append(words)
        labels.append(category_ids[row['category']])
    return (texts, tokens, torch.tensor(labels), categories, len(rows))

texts, tokens, labels, categories, original_count = read_training_data(TRAIN_PATH, settings.max_length)
print(f'Original CSV rows: {original_count:,}')
print(f'Distinct model inputs: {len(texts):,}')
print(f'Repeated inputs set aside: {original_count - len(texts)}')
print('Category order:', categories)

Original CSV rows: 1,000
Distinct model inputs: 970
Repeated inputs set aside: 30
Category order: ['business', 'entertainment', 'sport', 'tech']


# 3. Split the data

To train the model, I split the data first to check its predictions. I also made a seperate validation group for choosing the exact epoch count.

```text
970 distinct inputs
    ├── 776 development articles
    │     ├── 621 training articles: learn weights
    │     └── 155 validation articles: choose epoch count
    └── 194 holdout articles: check the chosen approach
```

The whole split_data function simply groups the articles by their category, shuffles each group, and then sets aside about 20% of each to make a stratified split to keep the category proportions similar. The function is called 2 times to mirror the split I showed above.

In [28]:
def split_data(indices, labels, seed):
    rng = random.Random(seed)
    answers = labels.tolist()
    training = []
    holdout = []
    # Split each category separately to keep a similar category mix.
    for label in sorted(set(answers)):
        group = [index for index in indices if answers[index] == label]
        rng.shuffle(group)
        count = round(0.2 * len(group))
        holdout.extend(group[:count])
        training.extend(group[count:])
    rng.shuffle(training)
    rng.shuffle(holdout)
    return (torch.tensor(training), torch.tensor(holdout))

development, holdout = split_data(list(range(len(texts))), labels, settings.seed)
training, validation = split_data(development.tolist(), labels, settings.seed + 1)
assert not set(development.tolist()) & set(holdout.tolist())
assert not set(training.tolist()) & set(validation.tolist())
print(f'Learn weights: {len(training)}; choose epoch count: {len(validation)}; check: {len(holdout)}')


Learn weights: 621; choose epoch count: 155; check: 194


# 4. Convert words into IDs

Since networks can't receive strings directly, I made it a dictionary that assigns each known word an integer ID. For instance, suppose it has:


```python
{'<PAD>': 0, '<UNK>': 1, 'team': 2, 'won': 3}
```

Then a short article could be encoded as:

```text
['team', 'won'] → [2, 3, 0, 0, ...]
```

ID **0** fills all of the unused positions while ID **1** represents a missing word from the collected vocabulary. `Counter` counts word occurrences wherein words that appear at least twice can receive their own IDS. `Encode` starts with a zero-filled tensor and writes each article's IDs into its row. For 621 training articles then the shape becomes: `(621, 128)`. Note that strictly only the 622 training articles determine this first vocabulary.

In [29]:
def build_vocabulary(token_lists, max_vocabulary):
    counts = Counter((word for words in token_lists for word in words))
    # Reserve these IDs before adding ordinary words.
    vocabulary = {'<PAD>': 0, '<UNK>': 1}
    ordered = sorted(counts.items(), key=lambda item: (-item[1], item[0]))
    for word, count in ordered:
        if count >= 2 and len(vocabulary) < max_vocabulary:
            vocabulary[word] = len(vocabulary)
    return vocabulary

def encode(token_lists, vocabulary, max_length):
    # Start with padding everywhere, then fill each row with word IDs.
    encoded = torch.zeros(len(token_lists), max_length, dtype=torch.long)
    for index, words in enumerate(token_lists):
        # Unknown words use ID 1; use one unknown token if the list is empty.
        ids = [vocabulary.get(word, 1) for word in words[:max_length]] or [1]
        encoded[index, :len(ids)] = torch.tensor(ids)
    return encoded

training_tokens = [tokens[index] for index in training.tolist()]
training_vocabulary = build_vocabulary(training_tokens, settings.max_vocabulary)
training_features = encode(training_tokens, training_vocabulary, settings.max_length)
validation_features = encode([tokens[index] for index in validation.tolist()], training_vocabulary, settings.max_length)
print('Vocabulary entries for the first training stage:', len(training_vocabulary))
print('Training input shape:', tuple(training_features.shape))


Vocabulary entries for the first training stage: 5390
Training input shape: (621, 128)


# 5. Creating the model

As mentioned, I used a word embedding and an LSTM to predict whether an article is about **business, entertainment, sport, or tech**.

1. The article is converted to lowercase words, keeping the first 128. Each word is replaced with its vocabulary ID. Unknown words use ID `1` and unused positions use padding ID `0` as. explained in the previous step

2. The embedding layer converts each word ID into 64 numbers wherein these numbers are adjusted during training to help the model predict topics.

3. The LSTM then processes the word representations in order, carrying information from earlier words. It produces 64 output values at each position.

4. I average the LSTM outputs for the real words to get one list of 64 numbers for the article. I also excluded some padding so during training, dropout temporarily zeros some values to reduce reliance on individual values.

5. Last, a linear layer converts the article representation into four scores, one per category and then the highest score determines the predicted topic.

In [30]:
class MyModel(nn.Module):

    def __init__(self, vocabulary_size, number_of_categories):
        super().__init__()
        # One trainable 64-number vector for each vocabulary entry.
        self.embedding = nn.Embedding(vocabulary_size, 64, padding_idx=0)
        self.lstm = nn.LSTM(64, 64, batch_first=True)
        self.dropout = nn.Dropout(0.2)
        self.fc1 = nn.Linear(64, number_of_categories)

    def forward(self, x):
        embedded = self.embedding(x)
        output, _ = self.lstm(embedded)
        # Keep real-word positions when averaging the LSTM outputs.
        mask = (x != 0).unsqueeze(-1)
        pooled = (output * mask).sum(dim=1) / mask.sum(dim=1).clamp_min(1)
        # Dropout operates during training and is disabled by model.eval().
        pooled = self.dropout(pooled)
        return self.fc1(pooled)

example_model = MyModel(len(training_vocabulary), len(categories))
with torch.no_grad():
    example_scores = example_model(training_features[:2])

## 6. Evaluation and training

I check the model by choosing the category with the highest score for each article.

I use two measurements:
- **Accuracy:** the fraction of articles predicted correctly.
- **Macro F1:** the average F1 score across the four categories, giving each category equal importance. 

To train the model, I shuffle the articles and process them in batches and then for each batch, the model predicts scores, cross-entropy loss compares them with the correct answers, and Adam adjusts the weights to reduce the loss. I also limit large gradients and use a small weight penalty to help control training.

After each epoch, I check the validation macro F1 and then select the epoch count with the highest score using lower validation loss to break ties.

In [31]:
@torch.no_grad()
def predict_scores(model, features, batch_size):
    model.eval()
    scores = []
    for start in range(0, len(features), batch_size):
        scores.append(model(features[start:start + batch_size]))
    return torch.cat(scores)


def classification_metrics(labels, predictions, categories):
    f1_scores = []
    for index in range(len(categories)):
        actual = labels == index
        predicted = predictions == index
        tp = (actual & predicted).sum().item()
        fp = (~actual & predicted).sum().item()
        fn = (actual & ~predicted).sum().item()
        denominator = 2 * tp + fp + fn
        f1_scores.append(2 * tp / denominator if denominator else 0.0)

    correct = (labels == predictions).sum().item()
    return {
        'macro_f1': sum(f1_scores) / len(categories),
        'accuracy': (labels == predictions).float().mean().item(),
        'correct': correct,
        'total': len(labels),
    }


def evaluate(model, features, labels, categories, batch_size):
    scores = predict_scores(model, features, batch_size)
    # Select the highest-scoring class separately for each article.
    predictions = scores.argmax(dim=1)
    metrics = classification_metrics(labels, predictions, categories)
    metrics['cross_entropy'] = nn.CrossEntropyLoss()(scores, labels).item()
    return metrics


def fit(features, labels, vocabulary_size, categories, epochs, settings, validation=None):
    # Every call starts a fresh model, rather than continuing a previous one.
    torch.manual_seed(settings.seed)
    model = MyModel(vocabulary_size, len(categories))
    optimizer = torch.optim.Adam(
        model.parameters(), lr=settings.learning_rate, weight_decay=0.0001
    )
    loss_function = nn.CrossEntropyLoss()
    generator = torch.Generator().manual_seed(settings.seed + 1000)
    best_key = None
    best_epoch = epochs
    best_metrics = None

    for epoch in range(1, epochs + 1):
        model.train()
        # Shuffle row positions while keeping each article paired with its label.
        order = torch.randperm(len(features), generator=generator)
        total_loss = 0.0

        for start in range(0, len(features), settings.batch_size):
            batch = order[start:start + settings.batch_size]
            # Clear the previous batch's gradients before calculating new ones.
            optimizer.zero_grad(set_to_none=True)
            scores = model(features[batch])
            loss = loss_function(scores, labels[batch])
            # Calculate gradients, limit their size, then update parameters.
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            total_loss += loss.item() * len(batch)

        # Validation selects the epoch count without changing any weights.
        if validation is not None:
            metrics = evaluate(model, *validation, categories, settings.batch_size)
            key = (metrics['macro_f1'], -metrics['cross_entropy'])
            if best_key is None or key > best_key:
                best_key = key
                best_epoch = epoch
                best_metrics = metrics

        if epoch == 1 or epoch % 5 == 0 or epoch == epochs:
            message = f'Epoch {epoch}/{epochs}: training loss={total_loss / len(features):.6f}'
            if validation is not None:
                message += f"; validation macro F1={metrics['macro_f1']:.6f}"
            print(message, flush=True)

    model.eval()
    return model, best_epoch, best_metrics


In [32]:
print('Choosing the epoch count using only validation macro F1:')
# Only keep the chosen epoch and validation metrics from this model.
_, chosen_epoch, chosen_metrics = fit(
    training_features, labels[training], len(training_vocabulary), categories,
    settings.epochs, settings, validation=(validation_features, labels[validation]),
)
print(f'Selected epoch: {chosen_epoch}')
print(f'Validation macro F1: {chosen_metrics["macro_f1"]:.6f}')


Choosing the epoch count using only validation macro F1:
Epoch 1/8: training loss=1.388105; validation macro F1=0.290474
Epoch 5/8: training loss=1.289851; validation macro F1=0.317718
Epoch 8/8: training loss=0.947527; validation macro F1=0.478266
Selected epoch: 8
Validation macro F1: 0.478266


# 7. Evaluate the holdout

Validation selected 30 epochs so now a fresh a model is trained for that long using all 777 development. A new vocabulary is also built from those articles because they are all training data for this stage. After that, the 194 holdout articles are encoded using this vocabulary and then passed through the trained model. Their labels are then used to count errors and calculate scores, but they do not update the vocabulary or weights.


The baseline always predicts the most common development category. `torch.bincount` counts the category IDs, `argmax()` finds the most common one and `torch.full_like` creates that same prediction for every holdout article. This provides a simple comparison for the trained network.

In [33]:
# Rebuild the dictionary from all articles used to train this evaluation model.
development_tokens = [tokens[index] for index in development.tolist()]
evaluation_vocabulary = build_vocabulary(development_tokens, settings.max_vocabulary)
development_features = encode(development_tokens, evaluation_vocabulary, settings.max_length)
holdout_features = encode(
    [tokens[index] for index in holdout.tolist()], evaluation_vocabulary, settings.max_length
)

evaluation_model, _, _ = fit(
    development_features, labels[development], len(evaluation_vocabulary),
    categories, chosen_epoch, settings
)
holdout_metrics = evaluate(
    evaluation_model, holdout_features, labels[holdout], categories, settings.batch_size
)

# A baseline makes one constant guess based on development labels only.
majority_label = torch.bincount(labels[development], minlength=len(categories)).argmax().item()
majority_predictions = torch.full_like(labels[holdout], majority_label)
baseline_metrics = classification_metrics(labels[holdout], majority_predictions, categories)

print(f"Holdout correct: {holdout_metrics['correct']}/{holdout_metrics['total']}")
print(f"Holdout accuracy: {holdout_metrics['accuracy']:.2%}")
print(f"Holdout macro F1: {holdout_metrics['macro_f1']:.6f}")
print(f"Baseline (always {categories[majority_label]}): "
      f"accuracy={baseline_metrics['accuracy']:.2%}, macro F1={baseline_metrics['macro_f1']:.6f}")


Epoch 1/8: training loss=1.378326
Epoch 5/8: training loss=1.228691
Epoch 8/8: training loss=0.886438
Holdout correct: 105/194
Holdout accuracy: 54.12%
Holdout macro F1: 0.494077
Baseline (always business): accuracy=29.38%, macro F1=0.113546
